<a href="https://colab.research.google.com/github/Innovatewithapple/BioasqRAG/blob/main/bioSAQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install rank-bm25

In [ ]:
!pip install transformers datasets

In [ ]:
import torch.nn as nn
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM,AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
from tqdm import tqdm
import torch.nn.functional as F
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.tokenize import sent_tokenize
import pandas as pd
from rank_bm25 import BM25Okapi
import gc
from transformers import BitsAndBytesConfig

In [ ]:
text_corpus = load_dataset('enelpol/rag-mini-bioasq','text-corpus')
question_answer_passage = load_dataset('enelpol/rag-mini-bioasq','question-answer-passages')

In [ ]:
text_corpus

DatasetDict({
    test: Dataset({
        features: ['passage', 'id'],
        num_rows: 40181
    })
})

In [ ]:
corpus_data = text_corpus['test']
corpus_data

Dataset({
    features: ['passage', 'id'],
    num_rows: 40181
})

In [ ]:
all_sentences = []

for data in corpus_data:
    sentences = sent_tokenize(data['passage'])

    for sentence in sentences:
        all_sentences.append(sentence)

with open("all_sentences.txt", "w", encoding="utf-8") as f:
    for sentence in all_sentences:
        f.write(sentence + "\n\n")

In [ ]:
rows = []

for data in corpus_data:

    passage_id = data['id']

    sentences = sent_tokenize(data['passage'])

    for sentence_idx, sentence in enumerate(sentences):

        rows.append({
            'passage_id': passage_id,
            'sentence_id': sentence_idx,
            'sentence': sentence
        })

df = pd.DataFrame(rows)

df.to_csv(
    "sentence_analysis_with_ids.csv",
    index=False
)

In [ ]:
def ExtractDataAndCreateChunks(chunksize, overlapsize):
    chunks = []
    for data in corpus_data:
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', data['passage'])

        current_chunk = []
        current_length = 0

        for sentence in sentences:

            sentence_words = sentence.split()
            sentence_length = len(sentence_words)

            # If adding sentence exceeds chunk size
            if current_length + sentence_length > chunksize:

                chunk_text = " ".join(current_chunk)

                chunks.append({
                    'original_passage_id': data['id'],
                    'chunk_text': chunk_text
                })

                # overlap handling
                overlap_sentences = current_chunk[-overlapsize:]

                current_chunk = overlap_sentences
                current_length = sum(len(sentence.split()) for sentence in current_chunk)

            current_chunk.append(sentence)
            current_length += sentence_length

        # Add remaining chunk
        if current_chunk:

            chunk_text = " ".join(current_chunk)

            chunks.append({
                'original_passage_id': data['id'],
                'chunk_text': chunk_text
            })

    return chunks
knowledge_strings = ExtractDataAndCreateChunks(chunksize=100,overlapsize=1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# **Encoder Model**

In [ ]:
# encoder_token_model = AutoTokenizer.from_pretrained('BAAI/bge-m3')
# encoder_token_model.vocab_size

In [ ]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [ ]:
# Model 1: for passages (replaces BGEWriter for database encoding)
class MedCPTPassageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained('ncbi/MedCPT-Article-Encoder')

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # CLS token, not mean pooling

# Model 2: for queries only
class MedCPTQueryEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained('ncbi/MedCPT-Query-Encoder')

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # CLS token

In [ ]:
passage_encoder = MedCPTPassageEncoder().to(device)
passage_encoder.eval()

query_encoder = MedCPTQueryEncoder().to(device)
query_encoder.eval()

# Two separate tokenizers too
from transformers import AutoTokenizer
passage_tokenizer = AutoTokenizer.from_pretrained('ncbi/MedCPT-Article-Encoder')
query_tokenizer = AutoTokenizer.from_pretrained('ncbi/MedCPT-Query-Encoder')

In [ ]:
knowledge_text = [i['chunk_text'] for i in knowledge_strings]
encoder_tokenised_data = passage_tokenizer(text=knowledge_text,padding=True,max_length=256,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
encoder_data = RAGKnowledgeDataset(encoder_tokenised_data)
encoder_Data_Loader = DataLoader(dataset=encoder_data,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

In [ ]:
# class BGEWriter(nn.Module):
#   def __init__(self) -> None:
#     super().__init__()
#     self.bge = AutoModel.from_pretrained('BAAI/bge-m3')

#   def forward(self,input_ids,attention_mask):
#     bgeOutput = self.bge(input_ids=input_ids,attention_mask=attention_mask)
#     bgeOutput = bgeOutput.last_hidden_state
#     mean = attention_mask.unsqueeze(-1).float()
#     mean = (bgeOutput*mean).sum(dim=1)/mean.sum(dim=1)
#     return mean

In [ ]:
# encoder_Model = BGEWriter().to(device)
# encoder_Model.eval()

In [ ]:
# database_vectors = []
# with torch.no_grad():
#   with torch.amp.autocast('cuda'):
#     for batch in tqdm(encoder_Data_Loader,desc='Encoding Model'):
#       input_ids,attention_mask = batch['input_ids'].to(device),batch['attention_mask'].to(device)

#       model_outputs = encoder_Model(input_ids,attention_mask)
#       database_vectors.append(model_outputs)

# vector_database = torch.cat(database_vectors,dim=0).cpu()
# print(vector_database.shape)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
database_vectors = []
with torch.no_grad():
    with torch.amp.autocast('cuda'):
        for batch in tqdm(encoder_Data_Loader, desc='Encoding passages'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            output = passage_encoder(input_ids, attention_mask)  # passage encoder
            database_vectors.append(output.cpu())

vector_database = torch.cat(database_vectors, dim=0)
del output

gc.collect()
torch.cuda.empty_cache()

Encoding passages: 100%|██████████| 4077/4077 [06:36<00:00, 10.28it/s]


In [ ]:
train_quesAnsPassage = question_answer_passage['train']
train_quesAnsPassage

Dataset({
    features: ['question', 'answer', 'id', 'relevant_passage_ids'],
    num_rows: 4012
})

In [ ]:
Reranking_token_Model = AutoTokenizer.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model = AutoModelForSequenceClassification.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model.eval()

In [ ]:

del decoder_tokenizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
decoder_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
decoder_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct',
    dtype=torch.float16,
    device_map='auto'
)

In [ ]:
def GetFinalAnswer(input_ids_decoder,attention_mask):
  with torch.no_grad():
   with torch.amp.autocast('cuda'):
    output_ids = decoder_Model.generate(
        input_ids_decoder,
        attention_mask=attention_mask,
        max_new_tokens=100,       # Gives it plenty of room to write a complete sentence [prev]
        do_sample=True,          # Enables natural conversational sampling [prev]
        top_k=10,                # Restricts the word pool to keep it highly focused [prev]
        temperature=0.2,         # Drops creativity near zero to lock onto context facts [prev]
        repetition_penalty=1.15, # Nudges it away from loops without breaking grammar rules [prev]
        no_repeat_ngram_size=0,  # Turned off: Disables the rigid, destructive phrase bans completely [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

   newTokens = output_ids[0,input_ids_decoder.shape[1]:]
   final_output = decoder_tokenizer.decode(newTokens,skip_special_token=True)

   print('FinalAnswer:', final_output)

In [ ]:
def ManageDecoderModel(finalChunk, question):
    messages = [                          # indent this
        {
            "role": "system",
            "content": """You are a biomedical expert answering questions based on scientific literature.

Instructions:
- Synthesize information from ALL provided context passages
- Focus on specific biological mechanisms, markers, findings and specifically the names
- Give a direct answer in 1-2 sentence mentioning all specific histone markers and mechanisms found across the context passages.
- Use proper biomedical terminology
- Find detailed information from the context starting to end
- Do not add information beyond what is in the context"""
        },
        {
            "role": "user",
            "content": f"""Context passages:
{finalChunk}

Question: {question}

Answer:"""
        }
    ]

    text_promt = decoder_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print("\ntext_promt: ", text_promt)
    input_ids_decoder = decoder_tokenizer.encode(text_promt, return_tensors='pt').to(device)
    attention_mask = torch.ones_like(input_ids_decoder).to(device)
    GetFinalAnswer(input_ids_decoder=input_ids_decoder, attention_mask=attention_mask)

In [ ]:
def rerankedForm(question,retrieval_chunks,encoder_scores):
  pairs = []
  all_chunks = [i['chunk_text'] for i in retrieval_chunks]
  originalids = [i['original_passage_id'] for i in retrieval_chunks]
  for chunk in all_chunks:
    pairs.append([question,chunk])

  Reranking_tokenize_query = Reranking_token_Model(text=pairs,max_length=512,padding=True,truncation=True,return_tensors='pt')

  with torch.no_grad():
    with torch.amp.autocast('cuda'):
      reranking_model_output = rerank_model(Reranking_tokenize_query['input_ids'],Reranking_tokenize_query['attention_mask'])
      rerank_scores = reranking_model_output.logits.squeeze()

  # normalize encoder scores for top_k results
  encoder_scores = top_scores[0]  # scores from your similarity computation
  encoder_norm = (encoder_scores - encoder_scores.min()) / (encoder_scores.max() - encoder_scores.min())

  # normalize reranker scores
  rerank_norm = (rerank_scores - rerank_scores.min()) / (rerank_scores.max() - rerank_scores.min())

  # combine both
  final_scores = 0.4 * encoder_norm + 0.6 * rerank_norm
  sorted_indics = torch.argsort(final_scores, descending=True)
  # print('\n\nSorted_indics:',sorted_indics)
  print("\n===== RERANKED RESULTS =====\n")

  #Rerank the chunks again
  RerankedIds = []
  for i,idx in enumerate(sorted_indics):
    # print(f"Rank {i+1}")
    # print(f"Rerank Score: {rerank_scores[idx].item():.4f}")
    # print("-" * 50)
    # print(retrieval_chunks[idx])
    # print("\n")
    RerankedIds.append(originalids[idx])
  final_chunks = [
    all_chunks[idx]
    for idx in sorted_indics[:5]
  ]
  print('\nRerankedIDS: ',RerankedIds)
  final_context = "\n\n".join(final_chunks)
  ManageDecoderModel(finalChunk=final_context,question=question)

In [ ]:
for quesIdx,i in enumerate(train_quesAnsPassage['question'][1]):
  retrieval_chunks = []
  #Tokenize
  queryQ = i
  query_tokenize = query_tokenizer(text=queryQ,padding='max_length',max_length=64,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt').to(device)

  #Embedding
  with torch.no_grad():
    with torch.amp.autocast('cuda'):
      query_model_output = query_encoder(query_tokenize['input_ids'],query_tokenize['attention_mask'])
  query_vector_cpu = query_model_output.cpu()

  #matrix multiplaction
  vector_database_normalize = F.normalize(vector_database,p=2,dim=-1)
  vector_query_normalize = F.normalize(query_vector_cpu,p=2,dim=-1)

  similarity_score = vector_query_normalize @ vector_database_normalize.T
  # ---- DIAGNOSTIC: ADD HERE ----
  target_ids = [23179372, 19270706, 23184418]
  all_scores_sorted = similarity_score[0].argsort(descending=True)  # sort all chunks

  for rank, idx in enumerate(all_scores_sorted):
    chunk_id = knowledge_strings[idx]['original_passage_id']
    if chunk_id in target_ids:
        print(f"Rank {rank} | score={similarity_score[0][idx]:.4f} | passage_id={chunk_id}")
  # ---- END DIAGNOSTIC ----
  #Get top 3 scores:
  top_k=7
  top_scores,top_indics = torch.topk(input=similarity_score,k=top_k,dim=-1)
  print(f'======Matching=====')
  print(f'top_scores: {top_scores} | top_indexes: {top_indics}')

  print(f'========'*50)
  print(f'\n\nQuestion:{quesIdx}: {i}')
  print(f'\n\nExpected Answer: {train_quesAnsPassage['answer'][quesIdx]}')
  print(f'\n\nExpected AnswerID: {train_quesAnsPassage['relevant_passage_ids'][quesIdx]}')
  savedids = []
  for idx in top_indics[0]: # here put 0 because top_indics has 2D array [[1,2,3,4,5]] kinda
   # print(f'\nRetrived-AnswerID: {knowledge_strings[idx]['original_passage_id']}')
   retrieval_chunks.append(knowledge_strings[idx])
   savedids.append(knowledge_strings[idx]['original_passage_id'])
  print("Retrive-IDS: ",savedids)
  rerankedForm(question=i,retrieval_chunks=retrieval_chunks,encoder_scores=top_scores[0].cpu())

Rank 4203 | score=0.7655 | passage_id=23184418
Rank 5461 | score=0.7626 | passage_id=23184418
Rank 19420 | score=0.7453 | passage_id=19270706
Rank 20063 | score=0.7447 | passage_id=19270706
Rank 27029 | score=0.7392 | passage_id=23184418
Rank 29808 | score=0.7373 | passage_id=23179372
Rank 36766 | score=0.7327 | passage_id=23179372
Rank 46335 | score=0.7268 | passage_id=23184418
Rank 58660 | score=0.7194 | passage_id=19270706
Rank 62049 | score=0.7174 | passage_id=23179372
Rank 126713 | score=0.6553 | passage_id=23179372
======Matching=====
top_scores: tensor([[0.8265, 0.8209, 0.8200, 0.8195, 0.8185, 0.8135, 0.8128]]) | top_indexes: tensor([[125137, 125135, 125136, 125138, 124958, 124959, 105802]])


Question:0: W


Expected Answer: Aberrant patterns of H3K4, H3K9, and H3K27 histone lysine methylation were shown to result in histone code alterations, which induce changes in gene expression, and affect the proliferation rate of cells in medulloblastoma.


Expected AnswerID: [23179372, 1

In [ ]:
# find and print the best chunk from this passage
for chunk in knowledge_strings:
    if chunk['original_passage_id'] == 23184418:
        print(chunk['chunk_text'])
        break

Recent sequencing efforts have described the mutational landscape of the 
pediatric brain tumor medulloblastoma. Although MLL2 is among the most frequent 
somatic single nucleotide variants (SNV), the clinical and biological 
significance of these mutations remains uncharacterized. Through targeted 
re-sequencing, we identified mutations of MLL2 in 8 % (14/175) of MBs, the 
majority of which were loss of function. Notably, we also report mutations 
affecting the MLL2-binding partner KDM6A, in 4 % (7/175) of tumors.
